Reference: https://www.youtube.com/watch?v=hAtRGwv4keY

# **Installing Required Packages**

In [ ]:
!pip install tiktoken
!pip install langchain
!pip install openai==0.28
!pip install PyPDF2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 810.5/810.5 kB 16.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 46.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.9/273.9 kB 28.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.9/86.9 kB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.8/144.8 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 24.0
    Uninstalling packaging-24.0:
      Successfully uninstalled packaging-24.0


In [ ]:
!pip install qdrant_client

# **Importing Required Packages**

In [ ]:
from PyPDF2 import PdfReader
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Qdrant
from langchain.embeddings import OpenAIEmbeddings
from qdrant_client import QdrantClient,models
from qdrant_client.http.models import PointStruct
import os
import uuid
import openai

In [ ]:
import requests
import tempfile


In [ ]:
openai.api_key = "yourkey"

In [ ]:
record = 0


from qdrant_client import QdrantClient

connection = QdrantClient(
    url="https://20553d32-b501-4759-87ee-001cd4d140eb.europe-west3-0.gcp.cloud.qdrant.io:6333",
    api_key="yourapikey",
)

In [ ]:
connection.recreate_collection(
    collection_name = "pdf_qa_chatbot_w_qdrant",
    vectors_config = models.VectorParams(size=1536, distance = models.Distance.COSINE)
)

True

In [ ]:
print("Create collection response:", connection)

Create collection response: <qdrant_client.qdrant_client.QdrantClient object at 0x7e7d2de5cc10>


In [ ]:
info = connection.get_collection(collection_name = "pdf_qa_chatbot_w_qdrant")


In [ ]:
print("Collection info:", info)

Collection info: status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> vectors_count=0 indexed_vectors_count=0 points_count=0 segments_count=2 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1536, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=None), wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0), quantization_config=None) payload_schema={}


In [ ]:
for get_info in info:
  print(get_info)

('status', <CollectionStatus.GREEN: 'green'>)
('optimizer_status', <OptimizersStatusOneOf.OK: 'ok'>)
('vectors_count', 0)
('indexed_vectors_count', 0)
('points_count', 0)
('segments_count', 2)
('config', CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1536, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=None), wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0), quantization_config=None))
('paylo

In [ ]:
def process_pdf(pdf_input, pdf_url):
    unique_filename = f"downloaded_pdf_{uuid.uuid4()}.pdf"
    filepath = os.path.join(tempfile.gettempdir(), unique_filename)

    if pdf_input is not None:
        # Assuming pdf_input is an uploaded file's content
        with open(filepath, "wb") as f:
            f.write(pdf_input.read())  # pdf_input should be a file-like object
    elif pdf_url:
        # Handle PDF URL download
        response = requests.get(pdf_url)
        if response.status_code != 200:
            return None, "Failed to download the PDF. Status code: " + str(response.status_code)
        with open(filepath, "wb") as f:
            f.write(response.content)
    else:
        return None, "No PDF provided."

    return filepath, None


In [ ]:
def read_data_from_pdf(filepath):
    text = ""
    with open(filepath, "rb") as file:
        pdf_reader = PdfReader(file)
        for page in pdf_reader.pages:
            text += page.extract_text() or ""
    return text

In [ ]:
# converting text into chunks
def get_text_chunks(text):
  text_splitter = CharacterTextSplitter(
      separator = "\n",
      chunk_size = 1000,
      chunk_overlap = 100,
      length_function = len
  )
  chunks = text_splitter.split_text(text)
  return chunks

In [ ]:
# converting chunks to embeddings

def get_embedding(text_chunks, model_id = "text-embedding-ada-002"):
  points = []
  for idx, chunk in enumerate(text_chunks):
    response = openai.Embedding.create(
        input = chunk,
        model = model_id
    )
    embeddings = response["data"][0]["embedding"]
    point_id = str(uuid.uuid4())   # generate a unique ID for the point


    points.append(PointStruct(id = point_id, vector = embeddings, payload={"text": chunk}))

  return points

In [ ]:
# inserting data into qdrant database

def insert_data(get_points):
  operation_info = connection.upsert(
      collection_name = "pdf_qa_chatbot_w_qdrant",
      wait = True,
      points = get_points
  )

In [ ]:
def create_answer_with_context(query):
  response = openai.Embedding.create( input = query, model = "text-embedding-ada-002")
  embeddings = response["data"][0]["embedding"]
  search_result = connection.search(collection_name = "pdf_qa_chatbot_w_qdrant", query_vector = embeddings, limit = 1)
  prompt = "Context:\n"

  for result in search_result:
    prompt += result.payload["text"] + "\n--------\n"
  prompt += "Question:" + query + "\n------\n" + "Answer:"

  print("-------- PROMPT START --------")
  print(":", prompt)
  print("-------- PROMPT END ------ ")

  completion = openai.ChatCompletion.create(model = "gpt-3.5-turbo", messages = [{"role": "user", "content":prompt}])
  return completion.choices[0].message.content

In [ ]:
def main(pdf_input=None, pdf_url=None, question="What is the abstract of the patent?"):
    # Process the PDF and get the filepath
    filepath, error_message = process_pdf(pdf_input, pdf_url)
    if error_message:
        print(error_message)  # Handle the error message appropriately
        return

    if filepath:
        # Extract text from the processed PDF
        get_raw_text = read_data_from_pdf(filepath)
        # Split the extracted text into manageable chunks
        chunks = get_text_chunks(get_raw_text)
        # Convert text chunks into embeddings
        vectors = get_embedding(chunks)
        # Insert the embeddings into the Qdrant database
        insert_data(vectors)
        # Generate an answer based on the provided query using the context from the PDF
        answer = create_answer_with_context(question)
        print(answer)
    else:
        print("Failed to process the PDF.")

# Example usage
if __name__ == "__main__":

    main(pdf_url="https://patentimages.storage.googleapis.com/27/52/78/438b39445c6d92/US9646356.pdf")


-------- PROMPT START --------
: Context:
(12) United States Patent 
 Schwie et al. USOO9646356B1 
 US 9,646,356 B1 
 May 9, 2017 (10) Patent No.: 
 (45) Date of Patent: 
 (54) SELF-DRIVING VEHICLE SYSTEMS AND 
 METHODS 
 (71) Applicants: Wesley Edward Schwie, Philadelphia, PA (US); Eric John Wengreen, 
 Sammamish, WA (US) 
 (72) Inventors: Wesley Edward Schwie, Philadelphia, PA (US); Eric John Wengreen, 
 Sammamish, WA (US) 
 (*) Notice: Subject to any disclaimer, the term of this patent is extended or adjusted under 35 U.S.C. 154(b) by 0 days. 
 (21) Appl. No.: 15/181,413 
 (22) Filed: Jun. 14, 2016 
 (51) Int. Cl. G06O 50/30 (2012.01) 
 G05D I/O (2006.01) 
 G05D L/12 (2006.01) 
 G08G L/00 (2006.01) 
 G06O 20/12 (2012.01) 
 (52) U.S. Cl. 
 CPC ........... G06O 50/30 (2013.01); G05D I/0022 
 (2013.01); G05D I/0027 (2013.01); G05D 
 I/0088 (2013.01); G05D 1/12 (2013.01); 
 G06O 20/12 (2013.01); G08G 1/202 (2013.01) 
 (58) Field of Classification Search 
 CPC ...... G06Q 50/30; G06Q 20/

# **Gradio App**

In [ ]:
!pip install gradio


In [ ]:
import gradio as gr


def process_and_query_pdf(pdf_file, pdf_url, query):
    if pdf_file is not None:
        # Adjusted to handle file uploads correctly with newer Gradio versions
        filepath = process_pdf(pdf_file, None)
    elif pdf_url:
        filepath = process_pdf(None, pdf_url)
    else:
        return "No PDF provided."

    if filepath:
        get_raw_text = read_data_from_pdf(filepath)
        chunks = get_text_chunks(get_raw_text)
        vectors = get_embedding(chunks)
        insert_data(vectors)
        answer = create_answer_with_context(query)
        return answer
    else:
        return "Failed to process PDF."

# Adjusted definition for Gradio interface
iface = gr.Interface(
    fn=process_and_query_pdf,
    inputs=[
        gr.File(label="Upload PDF File"),
        gr.Textbox(label="Or Enter PDF URL"),
        gr.Textbox(label="Enter your Query")
    ],
    outputs=gr.Textbox(label="Answer"),
    title="PDF Content Query with Qdrant",
    description="""Upload a PDF file or enter a PDF URL along with your query about the content of the PDF.
                   The system will process the PDF and try to provide an answer based on its content.""",
    examples=[
        [None, "https://patentimages.storage.googleapis.com/27/52/78/438b39445c6d92/US9646356.pdf", "What is the key invention in this?"],
        [None, "https://patentimages.storage.googleapis.com/27/52/78/438b39445c6d92/US9646356.pdf", "List keypoints"],
        [None, "https://patentimages.storage.googleapis.com/27/52/78/438b39445c6d92/US9646356.pdf", "Summarize please"]
    ]
)

# To run the app, call the launch() method
if __name__ == "__main__":
    iface.launch()


Setting queue=True in a Colab notebook requires sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://5b53f1a1ddbb6b5f7b.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
